**FAKE NEWS DETECTO**R

NAME: YISEHAK ALELIGN

GROUP 3


1. Import Libraries

In [12]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

2: Load Dataset

In [13]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

CHECKing COLUMNS

In [14]:
print("Fake columns:", fake.columns)
print("True columns:", true.columns)

Fake columns: Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')
True columns: Index(['version https://git-lfs.github.com/spec/v1'], dtype='object')


4: SAFE DATA PREPARATION

In [15]:
# Add labels
fake['label'] = 1
true['label'] = 0

# Combine datasets
df = pd.concat([fake, true])

# 🔥 Convert everything to string (avoids errors)
df = df.astype(str)

# 🔥 Combine ALL columns into one text column (THIS FIXES EVERYTHING)
df['text'] = df.apply(lambda row: ' '.join(row.values), axis=1)

# Keep only needed columns
df = df[['text', 'label']]

# Shuffle dataset
df = df.sample(frac=1).reset_index(drop=True)

df.head()

,text,label
0,oid sha256:ba0844414a65dc6ae7402b8eee5306da24b...,0
1,oid sha256:bebf8bcfe95678bf2c732bf413a2ce5f621...,1
2,size 62789876 1,1
3,size 53582940 0,0


5: Text Preprocessing (TF-IDF)

In [16]:
X = df['text']
y = df['label']

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(X).toarray()

6: Train-Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

7: Build Neural Network

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

model.add(Dense(128, activation='relu', input_shape=(X.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,345 (36.50 KB)

 Trainable params: 9,345 (36.50 KB)

 Non-trainable params: 0 (0.00 B)

8: Train Model

In [22]:
# Fix label type
df['label'] = df['label'].astype(int)

In [23]:
X = df['text'].astype(str)
y = df['label'].astype(int)

In [24]:
# Train the model
history = model.fit(
    X_train.astype('float32'),
    y_train.astype('float32'),
    epochs=10,
    batch_size=32,
    validation_split=0.1
)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.0000e+00 - loss: 0.7528 - val_accuracy: 1.0000 - val_loss: 0.6734
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.0000e+00 - loss: 0.7317 - val_accuracy: 1.0000 - val_loss: 0.6850
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.0000e+00 - loss: 0.7122 - val_accuracy: 0.0000e+00 - val_loss: 0.6965
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - accuracy: 0.5000 - loss: 0.6935 - val_accuracy: 0.0000e+00 - val_loss: 0.7070
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 1.0000 - loss: 0.6755 - val_accuracy: 0.0000e+00 - val_loss: 0.7171
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - accuracy: 1.0000 - loss: 0.6595 - val_accuracy: 0.0000e+00 - val_loss: 0.7277
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 1.0000 - loss: 0.6448 - val_accuracy: 0.0000e+00 - val_loss: 0.7382
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - accuracy: 1.0000 - loss: 0.6306 - val

9: Evaluate Model

In [27]:
# Fix data types before evaluation
X_test = X_test.astype('float32')
y_test = y_test.astype('float32')

# Evaluate model
loss, accuracy = model.evaluate(X_test, y_test)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step - accuracy: 0.0000e+00 - loss: 0.7713
Test Loss: 0.7713170051574707
Test Accuracy: 0.0


10: Test With New Headlines

In [21]:
def predict_news(text):
    text_vec = vectorizer.transform([text]).toarray()
    prediction = model.predict(text_vec)

    if prediction[0][0] > 0.5:
        return "Fake News"
    else:
        return "Real News"

print(predict_news("Scientists confirm water on Mars."))
print(predict_news("Secret government creates invisible humans."))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step
Real News
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Real News


11: Save Model

In [28]:
model.save("fake_news_model.h5")

Short Explanation



Neural Network Explanation

1. Architecture Used:

I used a Feedforward Neural Network with:

Input layer

Two hidden layers (128 neurons and 64 neurons)

Output layer with 1 neuron

2. Number of Epochs:

The model was trained for 10 epochs.

3. Activation Functions:

ReLU for hidden layers

Sigmoid for output layer

4. Model Accuracy:

The model achieved an accuracy of approximately (put your result here, e.g., 92%) on the test dataset.